# Institutional Convergence via Inverse Planning

**The Setup:** Three people working on a project together. Each agent has a random role preference. If they act according to their role preference, they get an intrinsic reward. Collective reward requires every action to be filled at each time step. 

**The Goal:** Through observing each other's actions over time, agents infer role preferences and converge on complementary roles.

**Key Components:**
1. **Forward Model (Q-function)**: Given role preferences, compute optimal actions
2. **Inverse Model**: Given observed actions, infer role preferences
3. **Convergence**: As beliefs about roles sharpen, stable leader-follower emerges

**To Add later**
1. Each has a latent role preference (LEADER or FOLLOWER), but they don't know each other's preferences.

**What I'm trying to demonstrate:** Have agents converge on the same institutional structure and differentiate themselves into different roles via an Imagined We style approach, despite initially not knowing that structure with certainty. e.g. it'd be interesting to model how organic leaders emerge in protest movements or even group discussions even when there's no initial hierarchy to begin with.  

In [40]:
# Import dependencies
from functools import cache
import jax
import jax.numpy as np
from memo import memo
from enum import IntEnum
import matplotlib.pyplot as plt

In [41]:
num_agents = 3
num_actions = 3
num_roles = 3


class ACTIONS(IntEnum):
    CODE = 0 
    WRITE = 1
    DESIGN = 2

class ROLES(IntEnum):
    CODER = 0
    WRITER = 1
    DESIGNER = 2
# discount factor
@jax.jit
def gamma():
    return 1.0

@jax.jit
def individual_reward(state, action, role):
    # If your action matches your preferred role, you get individual satisfaction
    return np.where(action == role, 1.0, 0.0)

@jax.jit
def collective_reward(state):
    # If all three roles are covered (each agent doing different task), team succeeds
    # state encodes which actions are being taken
    # For simplicity, reward when we have diversity of actions
    actions_in_state = state % (num_actions ** num_agents)
    action_counts = np.array([
        np.sum(actions_in_state == ACTIONS.CODE),
        np.sum(actions_in_state == ACTIONS.WRITE),
        np.sum(actions_in_state == ACTIONS.DESIGN)
    ])
    # Maximum collective reward when each action is taken by exactly one agent
    return np.float32(np.all(action_counts == 1))

# Assumes that the outcome of the actions are independent of the agent doing them
# If we want to change that, we need to have many more states and possibly discretize it over the agent's skills.
STATES = np.arange(num_agents * num_actions * num_roles)

In [42]:
@cache
@memo
def Q[state: STATES, action: ACTIONS, role: ROLES](t):
    alice: knows(state, action, role)
    alice: chooses(role_alice in ROLES, wpp=1)
    alice: chooses(action_alice in ACTIONS, to_maximize = 0.0 if t < 0 else Q[state, action, role](t-1))
    return E[
        collective_reward(state) + individual_reward(state, action, role) 
        + (0.0 if t < 0 else gamma() * Q[state, action, role](t-1))
    ]

|     return E[
|            ^
    


In [47]:
Q(5)

Array([[[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
        [0., 0., 7.]],

       [[7., 0., 0.],
        [0., 7., 0.],
    

In [44]:

@memo
def game[state: STATES, action: ACTIONS, role: ROLES](alice_pref, bob_pref, charlie_pref, t):
    # Initial knowledge 
    cast: [alice, bob, charlie]
    alice: knows(state, action, role)
    bob: knows(state, action, role)
    charlie: knows(state, action, role)
    
    # Choose initial role 
    alice: chooses(role_alice in ROLES, wpp=1)
    bob: chooses(role_bob in ROLES, wpp=1)
    charlie: chooses(role_charlie in ROLES, wpp=1)
    
    # Choose action based on their role 
    alice: chooses(action_alice in ACTIONS, wpp=1)
    bob: chooses(action_bob in ACTIONS, wpp=1)
    charlie: chooses(action_charlie in ACTIONS, wpp=1)


    
    return 1

In [45]:

# Payoff matrix: [my_action, other_action] -> reward
# Both asserting (conflict) is bad, both deferring (indecision) is bad
# One asserts, one defers (coordination) is good
PAYOFF_MATRIX = np.array([
    [0.0, 1.0],  # If I assert: get 0 if other asserts, get 1 if other defers
    [0.5, 0.5]   # If I defer: get 0.5 regardless (safe but suboptimal)
])

@jax.jit
def individual_payoff(action_alice, action_bob):
    return PAYOFF_MATRIX[action_alice, action_bob]

@jax.jit
def collective_payoff(action_alice, action_bob):
    return PAYOFF_MATRIX[action_alice, action_bob] + PAYOFF_MATRIX[action_bob, action_alice]

# Rationality parameter for action selection
@jax.jit
def beta():
    return 3.0

## Forward Model: Q-function

Given both agents' role preferences, compute the value of Alice taking an action.

Structure:
- Alice knows her own role preference
- Alice models Bob's behavior given his role preference
- Both agents act rationally to maximize their payoff

In [46]:
@cache
@memo
def Q_alice[role_pref: ROLES, action: ACTIONS](role_alice_pref, role_bob_pref, t):
    """
    Q-function for Alice given both agents' role preferences.
    Returns expected value of Alice taking 'action' when she prefers role_alice_pref
    and Bob prefers role_bob_pref.
    """
    # Alice knows her role preference
    alice: knows(role_alice_pref, action)
    
    # Bob has a role preference and acts accordingly
    bob: knows(role_bob_pref)
    
    # Bob chooses action rationally based on his preference
    # At depth 0, acts naively according to role preference
    # At depth > 0, acts rationally via softmax over Q-values
    bob: chooses(action_bob in ACTIONS,
                 wpp = (role_bob_pref == action_bob) if t <= 0 else 
                       exp(beta() * Q_bob[role_bob_pref, action_bob](role_bob_pref, role_alice_pref, t-1)))
    
    return E[individual_payoff(action, bob.action_bob)]

@cache
@memo
def Q_bob[role_pref: ROLES, action: ACTIONS](role_bob_pref, role_alice_pref, t):
    """
    Q-function for Bob (symmetric to Alice's).
    """
    # Bob knows his role preference
    bob: knows(role_bob_pref, action)
    
    # Alice has a role preference and acts accordingly
    alice: knows(role_alice_pref)
    
    # Alice chooses action rationally
    alice: chooses(action_alice in ACTIONS,
                   wpp = (role_alice_pref == action_alice) if t <= 0 else
                         exp(beta() * Q_alice[role_alice_pref, action_alice](role_alice_pref, role_bob_pref, t-1)))
    
    return E[individual_payoff(action, alice.action_alice)]

memo.core.MemoError: Knowing unknown choice
  file: "3187672997.py", line 10, in @memo Q_alice
        alice: knows(role_alice_pref, action)
               ^

  hint: observer does not yet model self's choice of role_alice_pref.
        So, it doesn't make sense for observer to model alice as
        knowing that choice.

  ctxt: This error was encountered in the frame of observer.  In that
        frame, observer is currently modeling the following 2 choices:
        role_pref, action.

  info: You are using memo 1.2.4, JAX 0.7.2, Python 3.12.11 on Darwin.


In [ ]:
# Test Q-function
print("Testing Q-function...")
print("\nQ-values at depth 0 (naive behavior):")
q0 = Q_alice(0)
print(f"Shape: {q0.shape}")  # [role_alice_pref, action, role_bob_pref]
print(f"\nAlice prefers LEADER, Bob prefers FOLLOWER:")
print(f"  Q(ASSERT) = {q0[ROLES.LEADER, ACTIONS.ASSERT, ROLES.FOLLOWER]:.3f}")
print(f"  Q(DEFER) = {q0[ROLES.LEADER, ACTIONS.DEFER, ROLES.FOLLOWER]:.3f}")

print("\nQ-values at depth 3 (rational behavior):")
q3 = Q_alice(3)
print(f"\nAlice prefers LEADER, Bob prefers FOLLOWER:")
print(f"  Q(ASSERT) = {q3[ROLES.LEADER, ACTIONS.ASSERT, ROLES.FOLLOWER]:.3f}")
print(f"  Q(DEFER) = {q3[ROLES.LEADER, ACTIONS.DEFER, ROLES.FOLLOWER]:.3f}")
print(f"\nAlice prefers LEADER, Bob prefers LEADER (conflict):")
print(f"  Q(ASSERT) = {q3[ROLES.LEADER, ACTIONS.ASSERT, ROLES.LEADER]:.3f}")
print(f"  Q(DEFER) = {q3[ROLES.LEADER, ACTIONS.DEFER, ROLES.LEADER]:.3f}")

## Inverse Model: Role Inference

Given an observed action, infer the agent's role preference.

Structure:
- Observer has uncertainty over the agent's role preference (prior)
- Observer models agent as acting rationally given their preference
- Observer conditions on the observed action
- Returns posterior probability over role preferences

In [ ]:
@memo
def infer_alice_role[role_pref: ROLES, action: ACTIONS](observed_action, role_bob_pref, t):
    """
    Infer Alice's role preference given her observed action.
    Assumes Bob's role preference is known (or conditioned on).
    """
    observer: knows(observed_action, role_bob_pref)
    
    # Observer's beliefs about Alice's role preference and behavior
    observer: thinks[
        alice: chooses(role_pref in ROLES, wpp = 1),  # Uniform prior over role preferences
        alice: chooses(action in ACTIONS,
                       wpp = exp(beta() * Q_alice[role_pref, action](role_pref, role_bob_pref, t)))
    ]
    
    # Condition on observed action
    observer: observes [alice.action] is observed_action
    
    # Return posterior over role preferences
    return observer[Pr[alice.role_pref == role_pref]]

@memo
def infer_bob_role[role_pref: ROLES, action: ACTIONS](observed_action, role_alice_pref, t):
    """
    Infer Bob's role preference given his observed action.
    Assumes Alice's role preference is known (or conditioned on).
    """
    observer: knows(observed_action, role_alice_pref)
    
    # Observer's beliefs about Bob's role preference and behavior
    observer: thinks[
        bob: chooses(role_pref in ROLES, wpp = 1),  # Uniform prior
        bob: chooses(action in ACTIONS,
                     wpp = exp(beta() * Q_bob[role_pref, action](role_pref, role_alice_pref, t)))
    ]
    
    # Condition on observed action
    observer: observes [bob.action] is observed_action
    
    # Return posterior over role preferences
    return observer[Pr[bob.role_pref == role_pref]]

In [ ]:
# Test inverse planning
print("Testing Inverse Planning...\n")

t_depth = 3

print(f"Scenario: Bob prefers FOLLOWER role\n")

print("If Alice takes ASSERT action:")
infer_assert = infer_alice_role(ACTIONS.ASSERT, ROLES.FOLLOWER, t_depth)
print(f"  P(Alice prefers LEADER | ASSERT) = {infer_assert[ROLES.LEADER]:.3f}")
print(f"  P(Alice prefers FOLLOWER | ASSERT) = {infer_assert[ROLES.FOLLOWER]:.3f}")

print("\nIf Alice takes DEFER action:")
infer_defer = infer_alice_role(ACTIONS.DEFER, ROLES.FOLLOWER, t_depth)
print(f"  P(Alice prefers LEADER | DEFER) = {infer_defer[ROLES.LEADER]:.3f}")
print(f"  P(Alice prefers FOLLOWER | DEFER) = {infer_defer[ROLES.FOLLOWER]:.3f}")

print("\n" + "="*60)
print(f"\nScenario: Bob prefers LEADER role (conflict scenario)\n")

print("If Alice takes ASSERT action:")
infer_assert_conflict = infer_alice_role(ACTIONS.ASSERT, ROLES.LEADER, t_depth)
print(f"  P(Alice prefers LEADER | ASSERT) = {infer_assert_conflict[ROLES.LEADER]:.3f}")
print(f"  P(Alice prefers FOLLOWER | ASSERT) = {infer_assert_conflict[ROLES.FOLLOWER]:.3f}")

print("\nIf Alice takes DEFER action:")
infer_defer_conflict = infer_alice_role(ACTIONS.DEFER, ROLES.LEADER, t_depth)
print(f"  P(Alice prefers LEADER | DEFER) = {infer_defer_conflict[ROLES.LEADER]:.3f}")
print(f"  P(Alice prefers FOLLOWER | DEFER) = {infer_defer_conflict[ROLES.FOLLOWER]:.3f}")

## Joint Inference: Mutual Role Inference

Both agents simultaneously infer each other's role preferences from observed actions.
This models the full symmetric inference problem.

In [ ]:
@memo
def joint_role_inference[role_alice: ROLES, role_bob: ROLES, 
                         action_alice: ACTIONS, action_bob: ACTIONS](obs_action_alice, obs_action_bob, t):
    """
    Joint inference over both agents' role preferences given both observed actions.
    Returns joint posterior P(role_alice, role_bob | action_alice, action_bob).
    """
    observer: knows(obs_action_alice, obs_action_bob)
    
    observer: thinks[
        # Prior over role preferences (uniform)
        alice: chooses(role_alice in ROLES, wpp = 1),
        bob: chooses(role_bob in ROLES, wpp = 1),
        
        # Both act rationally given their preferences
        alice: chooses(action_alice in ACTIONS,
                       wpp = exp(beta() * Q_alice[role_alice, action_alice](role_alice, role_bob, t))),
        bob: chooses(action_bob in ACTIONS,
                     wpp = exp(beta() * Q_bob[role_bob, action_bob](role_bob, role_alice, t)))
    ]
    
    # Condition on both observed actions
    observer: observes [alice.action_alice] is obs_action_alice
    observer: observes [bob.action_bob] is obs_action_bob
    
    return observer[Pr[(alice.role_alice == role_alice) & (bob.role_bob == role_bob)]]

In [ ]:
# Test joint inference
print("Testing Joint Role Inference...\n")

t_depth = 3

print("Scenario 1: Alice ASSERTS, Bob DEFERS (coordinated)")
joint1 = joint_role_inference(ACTIONS.ASSERT, ACTIONS.DEFER, t_depth)
print("\nJoint posterior P(role_alice, role_bob | actions):")
print(f"  P(Alice=LEADER, Bob=LEADER) = {joint1[ROLES.LEADER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=LEADER, Bob=FOLLOWER) = {joint1[ROLES.LEADER, ROLES.FOLLOWER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=LEADER) = {joint1[ROLES.FOLLOWER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=FOLLOWER) = {joint1[ROLES.FOLLOWER, ROLES.FOLLOWER]:.3f}")

print("\n" + "="*60)
print("\nScenario 2: Alice DEFERS, Bob ASSERTS (coordinated, reversed)")
joint2 = joint_role_inference(ACTIONS.DEFER, ACTIONS.ASSERT, t_depth)
print("\nJoint posterior P(role_alice, role_bob | actions):")
print(f"  P(Alice=LEADER, Bob=LEADER) = {joint2[ROLES.LEADER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=LEADER, Bob=FOLLOWER) = {joint2[ROLES.LEADER, ROLES.FOLLOWER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=LEADER) = {joint2[ROLES.FOLLOWER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=FOLLOWER) = {joint2[ROLES.FOLLOWER, ROLES.FOLLOWER]:.3f}")

print("\n" + "="*60)
print("\nScenario 3: Both ASSERT (conflict)")
joint3 = joint_role_inference(ACTIONS.ASSERT, ACTIONS.ASSERT, t_depth)
print("\nJoint posterior P(role_alice, role_bob | actions):")
print(f"  P(Alice=LEADER, Bob=LEADER) = {joint3[ROLES.LEADER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=LEADER, Bob=FOLLOWER) = {joint3[ROLES.LEADER, ROLES.FOLLOWER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=LEADER) = {joint3[ROLES.FOLLOWER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=FOLLOWER) = {joint3[ROLES.FOLLOWER, ROLES.FOLLOWER]:.3f}")

print("\n" + "="*60)
print("\nScenario 4: Both DEFER (indecision)")
joint4 = joint_role_inference(ACTIONS.DEFER, ACTIONS.DEFER, t_depth)
print("\nJoint posterior P(role_alice, role_bob | actions):")
print(f"  P(Alice=LEADER, Bob=LEADER) = {joint4[ROLES.LEADER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=LEADER, Bob=FOLLOWER) = {joint4[ROLES.LEADER, ROLES.FOLLOWER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=LEADER) = {joint4[ROLES.FOLLOWER, ROLES.LEADER]:.3f}")
print(f"  P(Alice=FOLLOWER, Bob=FOLLOWER) = {joint4[ROLES.FOLLOWER, ROLES.FOLLOWER]:.3f}")